In [1]:
# imports and configuration
import pandas as pd
import requests
import time
from pathlib import Path

## Functions

In [235]:
def load_tournament_results(filepath):
    tournament_results = pd.read_csv(filepath)
    print(f"loaded {len(tournament_results):,} records")
    print(f"unique cards: {tournament_results['card_name'].nunique():,}")
    return tournament_results
    
def drop_basic_land_rows(df):
    basic_lands = [
        'Plains', 'Island', 'Swamp', 'Mountain', 'Forest', 'Wastes',
        'Snow-Covered Plains', 'Snow-Covered Island', 'Snow-Covered Swamp', 
        'Snow-Covered Mountain', 'Snow-Covered Forest', 'Snow-Covered Wastes' #snow-covered wastes didn't exist at the time but just in case
    ]
    df = df[~df["card_name"].isin(basic_lands)]
    return df
    
def drop_unneeded_columns(df):
    drop_columns = ['record', 'player_wins' ,'player_losses', 'tournament_size_bucket'] #we ended up not using buckets
    df = df.drop(columns=drop_columns)
    return df

### Card score calculations ###
def calc_card_copy_score(df):
    # apply mainboard/sideboard weight to each copy
    df["card_copy_score"] = df.apply(
        lambda row: row["card_count"] * mainboard_weight if row["is_mainboard"]
        else row["card_count"] * sideboard_weight, axis=1
    )

    # sum weighted copies per card per player per tournament
    card_copy_scores = df.groupby(
        ["card_name", "player", "source_file"]
    )["card_copy_score"].sum().reset_index()
    #deck_scores.columns = ["card_name", "player", "source_file", "deck_copy_score"]
    max_possible_card_score = max_card_copies * mainboard_weight # this = 4 in our model
    card_copy_scores["card_copy_score"] = card_copy_scores["card_copy_score"] / max_possible_card_score
    return card_copy_scores
    
def calc_win_rate_score(df):
    win_rates = df[["player", "source_file", "win_rate"]].drop_duplicates()
    win_rates["win_rate_score"] = (win_rates["win_rate"] - 0.75) / (1.0 - 0.75)  # We omit any results under .75 so we need to renormalize it where a .75 is a 0
    win_rates = win_rates.drop(columns=['win_rate'])
    return win_rates

def calc_tournament_size_score(df):
    tournament_sizes = df[["source_file", "tournament_size"]].drop_duplicates()
    tournament_size_max = tournament_sizes["tournament_size"].max() 
    tournament_sizes["tournament_size_score"] = tournament_sizes["tournament_size"] / tournament_size_max # we do not log it as we want larger tournaments to skew the data
    tournament_sizes = tournament_sizes.drop(columns=["tournament_size"])
    return tournament_sizes

def aggregate_to_card_level(df):
    card_scores = df.groupby("card_name").agg(
        card_copy_score=("card_copy_score", "mean"),
        win_rate_score=("win_rate_score", "mean"),
        tournament_size_score=("tournament_size_score", "mean"),
    ).reset_index()
    return card_scores

def calc_tournament_consistency_score(df):
    tournament_counts = df[["card_name", "source_file"]].drop_duplicates()
    tournament_counts = tournament_counts.groupby("card_name")["source_file"].count().reset_index()
    tournament_counts = tournament_counts.rename(columns={"source_file": "unique_tournament_count"})
    max_tournaments = tournament_counts["unique_tournament_count"].max() # need to normalize most apperances as 1
    tournament_counts["tournament_consistency_score"] = tournament_counts["unique_tournament_count"] / max_tournaments
    tournament_counts = tournament_counts.drop(columns=["unique_tournament_count"])
    return tournament_counts

################
### Card weight calculations ###


### Misc ###
def show_cards(df, card_names):
    mask = df["card_name"].isin(card_names)
    return(df[mask].to_string(index=False))

## Code

### Universal Variables

In [114]:
data_dir = Path("C:/Users/david/Documents/GitHub/ai_projects/modern_cube/data")
filtered_data = data_dir / "processed/filtered_card_records_2016_2018.csv"
output_dir = data_dir / "processed"

# cube configuration
cube_size = 360

# weights for metrics
#As we do not have a way to know how often a sideboard card is used we have to guestimate
mainboard_weight = 1
sideboard_weight = .5
max_card_copies = 4 # max played in Modern (we omit basic lands)

# data window
data_start = "2016-01-01"
data_end = "2018-12-31"

### Executed Code

In [239]:
tournament_results = load_tournament_results(filtered_data)
tournament_results = drop_unneeded_columns(tournament_results)
tournament_results = drop_basic_land_rows(tournament_results)
print(len(tournament_results))
tournament_results.head()

loaded 118,070 records
unique cards: 1,633
110862


,card_name,card_count,is_mainboard,player,tournament_name,tournament_date,source_file,tournament_size,win_rate
0,Ancient Stirrings,4,True,BoozeMongoose,Modern Daily Swiss,2016-01-01T00:00:00Z,modern-daily-swiss-2016-01-019177191.json,24,1.0
1,Chromatic Sphere,4,True,BoozeMongoose,Modern Daily Swiss,2016-01-01T00:00:00Z,modern-daily-swiss-2016-01-019177191.json,24,1.0
2,Chromatic Star,4,True,BoozeMongoose,Modern Daily Swiss,2016-01-01T00:00:00Z,modern-daily-swiss-2016-01-019177191.json,24,1.0
3,"Emrakul, the Aeons Torn",1,True,BoozeMongoose,Modern Daily Swiss,2016-01-01T00:00:00Z,modern-daily-swiss-2016-01-019177191.json,24,1.0
4,Expedition Map,4,True,BoozeMongoose,Modern Daily Swiss,2016-01-01T00:00:00Z,modern-daily-swiss-2016-01-019177191.json,24,1.0


In [241]:
# calculate card metrics
card_metrics = calc_card_copy_score(tournament_results)

win_score = calc_win_rate_score(tournament_results)
card_metrics = pd.merge(card_metrics, win_score, on=["player", "source_file"], how="inner")
del win_score

tournament_size_score = calc_tournament_size_score(tournament_results)
card_metrics = pd.merge(card_metrics, tournament_size_score, on=["source_file"], how="inner")
del tournament_size_score

card_scores = aggregate_to_card_level(card_metrics)
del card_metrics

tournament_consistency_score = calc_tournament_consistency_score(tournament_results)
card_scores = pd.merge(card_scores, tournament_consistency_score, on=["card_name"], how="inner")
del tournament_consistency_score

print(f"card scores shape: {card_scores.shape}")
card_scores.head()

card scores shape: (1622, 5)


,card_name,card_copy_score,win_rate_score,tournament_size_score,tournament_consistency_score
0,Abbot of Keral Keep,0.625000,0.177778,0.079745,0.019912
1,Abrade,0.267287,0.142717,0.515525,0.101770
2,Abrupt Decay,0.521657,0.237490,0.173605,0.679204
3,Abundant Growth,0.583333,0.400000,0.126794,0.006637
4,Abzan Ascendancy,1.000000,0.200000,0.136364,0.002212
